# שיעור 10 - סוכני בינה מלאכותית בפרודקשן

בשיעור זה תלמדו **תבניות פרודקשן** לסוכני בינה מלאכותית באמצעות מסגרת העבודה Microsoft Agent (פייתון). אנו מכסים:

- **יכולת תצפית** — הוספת מדידת זמנים ורישום לאינטראקציות עם הסוכן
- **הערכה** — שימוש בסוכן מעריך לציון איכות התגובה
- **ניהול עלויות** — אסטרטגיות לאופטימיזציית טוקנים ולקביעת מודל

התרחיש הוא **סוכן נסיעות** המסייע למשתמשים לתכנן טיולים, עם ניטור והערכה המוטמעים מעל.


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## שיקולים בפרודקשן

העברת סוכני בינה מלאכותית מפרוטוטיפים לפרודקשן דורשת תשומת לב זהירה לשלושה עמודים:

1. **נראות** — יש צורך בראות לגבי מה שהסוכן עושה, כמה זמן זה לוקח ואילו כלים הוא קורא להם. ללא עקיבה ורישום, איתור כשלים בפרודקשן כמעט בלתי אפשרי.

2. **הערכה** — בדיקות איכות אוטומטיות מוודאות שהתגובות של הסוכן נשארות מדויקות, מלאות ומועילות לאורך זמן. סוכן מעריך יכול לתת ציונים לתגובות לפי קריטריונים מוגדרים.

3. **ניהול עלויות** — שימוש בטוקנים משפיע ישירות על העלות. אסטרטגיות כמו אופטימיזציית פקודות, בחירת מודלים ואחסון זמני עוזרות לשמור על הוצאות תחת שליטה מבלי להקריב איכות.


## בניית סוכן ניתן לצפייה

אנו מגדירים כלי נסיעה ועוטפים את קריאת הסוכן עם מדידת זמן כדי שנוכל לעקוב אחרי ההשהייה. בייצור הייתם משלבים את זה עם OpenTelemetry או מערכת מעקב דומה.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## דפוסי הערכה

דפוס נפוץ בייצור הוא להשתמש בסוכן שני כ**מעריך**. המעריך מדרג את תגובת הסוכן הראשי כנגד קריטריונים מוגדרים מראש כגון שלמות, דיוק ושימושיות.

זה מאפשר:
- שערי איכות אוטומטיים לפני שהתגובות מגיעות למשתמשים
- זיהוי נסיגה כאשר הפקודות או הדגמים משתנים
- ניטור מתמשך של ביצועי הסוכן לאורך זמן


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## אסטרטגיות ניהול עלויות

שליטה בעלויות היא קריטית לסוכני AI לייצור. הנה אסטרטגיות מפתח:

| אסטרטגיה | תיאור |
|---|---|
| **אופטימיזציית פקודות** | שמור על הוראות המערכת תמציתיות. הסר הקשר מיותר להפחתת טוקנים בקלט. |
    "| **בחירת מודל** | השתמש במודלים קטנים וזולים יותר (למשל GPT-4.1-mini) למשימות פשוטות כמו סיווג או חילוץ, והשאר מודלים גדולים יותר למשימות הסקת מסקנות מורכבות. |\n",
| **מטמון** | אחסן תוצאות כלי ושאילתות תכופות במטמון כדי למנוע קריאות API מיותרות. |
| **תקציבי טוקנים** | קבע מגבלות `max_tokens` כדי למנוע תגובות ארוכות לא צפויות. |
| **אצירת בקשות** | קבץ שאילתות משתמש מרובות לקריאה יחידה ל-API כאשר אפשרי. |

בפועל, שיטה מדרגת עובדת היטב: הפנה בקשות פשוטות למודל מהיר וזול והעבר רק שאילתות מורכבות למודל מתקדם יותר.


## סיכום

בשיעור זה למדת כיצד:

1. **להוסיף נראות** לאינטראקציות של הסוכן עם תזמון ורישום, ולבסס את התשתית למעקב וניטור.
2. **להעריך תגובות של סוכנים** באופן אוטומטי באמצעות סוכן שמעריך את השלמות, הדיוק והתועלת.
3. **לנהל עלויות** באמצעות אופטימיזציה של הפקודות, בחירת מודלים, מטמון, ותקציבי טוקנים.

דפוסי ייצור אלו עוזרים להבטיח שהסוכנים המלאכותיים שלך יהיו אמינים, מדידים וחסכוניים בעלות בקנה מידה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
